# validation check — KMeans → profitable clusters → LR (polynomial), DRAW market

The candidate pipeline for the `goals_foragst` space, draw market, now running on the shared
`evaluation.model_check` framework. Only the **space**, **target**, **segmenter** and **model**
cells below are specific to this check; the folds, ROI scoring, honest per-fold engine and the
three gates all come from the framework.

We also compare **two ways of drawing the cluster boundaries**:

- **running** — boundaries fit on the point-in-time (running) season averages: the state we
  actually see at decision time, noisier early in the season.
- **final** — boundaries fit on end-of-season stats (`team_season_final`), the stable "true
  character" of each team; matches are still ASSIGNED by their point-in-time stats. Same idea as
  the `cat3q` thresholds (boundaries from the full-season distribution, value placed against them
  point-in-time).

Only the boundaries differ. Every match is always assigned, and the model always fit, on running
features, so both variants are honest at bet time.

## Decision: use the `running` map

Both maps pass every gate, so the signal does not depend on how the boundaries are drawn
(map-invariant) — reassuring in itself. **`running` is the deployed choice** (logged in the
Summary): its league × season breakdown is less extreme, it is stronger on walk-forward, and it is
the state actually observable at bet time. `final` is kept below only as a robustness comparison.

In [1]:
# --- SETUP -------------------------------------------------------------------
# Load the wide ABT for one league group and pull in the model-check framework. Everything that
# never changes between checks (folds, ROI scoring, the honest per-fold engine, the three gates,
# the registry write) lives in `evaluation.model_check`; this notebook only defines the parts that
# DO change — the space, the target, the segmenter, and the model.
GROUP = "major"   # league group: which <group>/ ABT to validate
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

from evaluation.model_check import (
    run_check, evaluate, collect_bets, pooled_roi, roi_grouped,
    season_folds, expanding_folds,
)

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

# Attach each team's end-of-season goals for/against to every match it played, once as the home
# side and once as the away side. These columns feed only the "final" cluster-boundary variant; a
# match is never LOCATED by them (that would leak the season's outcome into its own features).
final_stats = pd.read_parquet(f"C:/Users/twluc/frame/01_data/02_features/{GROUP}/team_season_final.parquet")
final_stats = final_stats[["team", "season", "season_goals_for_avg", "season_goals_agst_avg"]]
home_final = final_stats.rename(columns={"team": "hmt_name",
    "season_goals_for_avg": "home_for_final", "season_goals_agst_avg": "home_agst_final"})
away_final = final_stats.rename(columns={"team": "awt_name",
    "season_goals_for_avg": "away_for_final", "season_goals_agst_avg": "away_agst_final"})
abt = abt.merge(home_final, on=["hmt_name", "season"], how="left").merge(away_final, on=["awt_name", "season"], how="left")

In [2]:
# --- SPACE (modifiable) ------------------------------------------------------
# The space is just the list of feature columns that describe the match. Swapping this list (and the
# dropna subset in ABT FILTERING) is all it takes to move the whole check to another space.
# RUNNING: each team's average so far this season, as known before the match (point-in-time).
RUNNING_FEATURES = [
    "hmt_season_goals_for_avg", "hmt_season_goals_agst_avg",
    "awt_season_goals_for_avg", "awt_season_goals_agst_avg",
]
# FINAL: the same four quantities from the finished season — used only to place the "final" boundaries.
FINAL_FEATURES = ["home_for_final", "home_agst_final", "away_for_final", "away_agst_final"]

In [3]:
# --- TARGET (modifiable) -----------------------------------------------------
# OUTCOME is the 0/1 event we bet on. IMPLIED is the price we actually get (b365) and is what the ROI
# is computed against. MRKT_IMPLIED is the margin-thinner "fair-ish" line used only to decide which
# segments are underpriced (in keep_profitable) — decide value against the fair line, bet at b365.
OUTCOME = "t_draw_flg"              # 1 when the match ended in a draw
IMPLIED = "b365_draw_impl"          # implied draw prob we bet against = 1 / b365 draw odds
MRKT_IMPLIED = "mrkt_draw_impl"     # market-consensus implied draw prob, for profitability only

In [4]:
# --- ABT FILTERING -----------------------------------------------------------
# Keep matches where both teams have enough history for stable season averages, and drop rows missing
# either the running OR the final features so both boundary variants are compared on the same matches.
MIN_GAME_NUMBER = 8
matches = abt[
    (abt["hmt_game_number"] > MIN_GAME_NUMBER)
    & (abt["awt_game_number"] > MIN_GAME_NUMBER)
].dropna(subset=RUNNING_FEATURES + FINAL_FEATURES + [IMPLIED, MRKT_IMPLIED, OUTCOME, "league", "season"]).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

matches: 5533   seasons: ['2223', '2324', '2425', '2526']   leagues: ['england', 'france', 'germany', 'italy', 'spain']


In [5]:
# --- SEGMENTER (modifiable) --------------------------------------------------
# A segmenter carves out the region we bet on. Its contract (see evaluation.model_check):
#     segmenter(train) -> labeler ;  labeler(rows) -> Series of labels, NaN = "do not bet here".
# It fits on the TRAIN fold only, then can label any rows (train or the held-out test).
#
# This one: fit KMeans boundaries on `boundary_features`, keep the clusters that beat the market on
# train (keep_profitable), and label each row by its kept cluster. Rows are always ASSIGNED by the
# RUNNING features, so a match placed by its point-in-time stats is honest even when the boundaries
# were drawn on "final" stats.

# Keep the segments (clusters / cells / categories) where the target happened MORE often than the
# market priced it — computed on the TRAIN fold only, so the held-out fold never influences the
# choice. This is the "profitable-segment" selection rule; a rule-based segmenter would skip it.
def keep_profitable(train, labels, min_matches=100):
    frame = pd.DataFrame({"label": labels.to_numpy(),
                          "won": train[OUTCOME].to_numpy(),
                          "implied": train[MRKT_IMPLIED].to_numpy()})
    stats = frame.groupby("label").agg(won=("won", "mean"), implied=("implied", "mean"), n=("won", "size"))
    return set(stats.index[(stats["n"] >= min_matches) & (stats["won"] > stats["implied"])])

def kmeans_profitable_segmenter(boundary_features, n_clusters, min_matches=100):
    def segmenter(train):
        km = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(train[boundary_features].to_numpy())
        train_clusters = pd.Series(km.predict(train[RUNNING_FEATURES].to_numpy()), index=train.index)
        kept = keep_profitable(train, train_clusters, min_matches)
        def labeler(rows):
            c = pd.Series(km.predict(rows[RUNNING_FEATURES].to_numpy()), index=rows.index)
            return c.where(c.isin(kept))     # non-kept clusters -> NaN -> not bet
        return labeler
    return segmenter

K = 5   # reference cluster count for the gates; k is swept separately as a knob-robustness diagnostic
SEGMENTERS = {
    "running": kmeans_profitable_segmenter(RUNNING_FEATURES, K),
    "final":   kmeans_profitable_segmenter(FINAL_FEATURES, K),
}

In [6]:
# --- MODEL (modifiable) ------------------------------------------------------
# The bet-signal model: given the space features, predict the probability of the target outcome.
# Today it is polynomial-expanded features (squares + pairwise products), standardised, then logistic
# regression. To try a different method (random forest, XGBoost, ...) swap this one factory — the
# only contract the framework needs is an sklearn-style estimator with .fit(X, y) and .predict_proba(X).
def make_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                         StandardScaler(), LogisticRegression(max_iter=3000))

BUFFER = 0.02   # only bet when the model probability beats the implied one by at least this margin

## Gates 1–3, running vs final
`evaluate()` runs the whole check (no registry write) so we can compare the two boundary maps. G1
LOSO (≥3/4 seasons), G2 drop-best-league, G3 walk-forward floor.

In [7]:
# evaluate() returns every number but neither prints nor logs — perfect for comparing variants
# without overwriting the registry row (only the Summary below logs, on the deployed 'running' map).
for name, seg in SEGMENTERS.items():
    r = evaluate(matches, features=RUNNING_FEATURES, outcome=OUTCOME, implied=IMPLIED,
                 segmenter=seg, make_model=make_model, buffer=BUFFER)
    if r["empty"]:
        print(f"[{name:7s}] no bets placed"); continue
    g = lambda ok: "PASS" if ok else "FAIL"
    print(f"[{name:7s}] pooled {r['pooled']:+.4f}  ({len(r['bets'])} bets)   "
          f"G1 {g(r['g1'])} (+{int((r['per_season'] > 0).sum())}/{len(r['per_season'])})   "
          f"G2 {g(r['g2'])} (drop {r['best_league']} -> {r['without_league'][r['best_league']]:+.3f})   "
          f"G3 {g(r['g3'])}")
    print("          seasons:  " + "  ".join(f"{s}:{v:+.3f}" for s, v in r["per_season"].items()))
    print("          walk-fwd: " + ("  ".join(f"{s}:{v:+.3f}" for s, v in r["wf"].items())
                                     if not r["walk"].empty else "no bets"))

[running] pooled +0.1356  (997 bets)   G1 PASS (+3/4)   G2 PASS (drop france -> +0.124)   G3 PASS
          seasons:  2223:+0.096  2324:+0.267  2425:-0.009  2526:+0.227
          walk-fwd: 2425:+0.059  2526:+0.227


[final  ] pooled +0.1904  (837 bets)   G1 PASS (+4/4)   G2 PASS (drop england -> +0.157)   G3 PASS
          seasons:  2223:+0.230  2324:+0.498  2425:+0.071  2526:+0.135
          walk-fwd: 2425:+0.046  2526:+0.135


## Knob robustness — sweep the cluster count k
Not a gate. The edge should survive a range of k; if it only appears at one setting it is fragile.

In [8]:
# Rebuild the segmenter per k (the factory makes this cheap) and read the pooled ROI. collect_bets is
# the framework's honest per-fold engine — same one the gates use, called here directly for the sweep.
print("pooled ROI (b365) by cluster count k:")
print("   k   running   final")
for k in [4, 5, 6, 8, 10]:
    common = dict(features=RUNNING_FEATURES, outcome=OUTCOME, implied=IMPLIED,
                  make_model=make_model, buffer=BUFFER)
    run = collect_bets(matches, season_folds(matches),
                       segmenter=kmeans_profitable_segmenter(RUNNING_FEATURES, k), **common)
    fin = collect_bets(matches, season_folds(matches),
                       segmenter=kmeans_profitable_segmenter(FINAL_FEATURES, k), **common)
    print(f"  {k:2d}   {pooled_roi(run['implied'], run['won']):+.4f}  {pooled_roi(fin['implied'], fin['won']):+.4f}")

pooled ROI (b365) by cluster count k:
   k   running   final


   4   +0.1103  +0.1325


   5   +0.1356  +0.1904


   6   +0.1307  +0.1652


   8   +0.1444  +0.1098


  10   +0.0594  +0.0705


## Diagnostic — where the edge lives (league × season), running map
Not a gate. Shows how concentrated / noisy the edge is at the finest granularity.

In [9]:
r = evaluate(matches, features=RUNNING_FEATURES, outcome=OUTCOME, implied=IMPLIED,
             segmenter=SEGMENTERS["running"], make_model=make_model, buffer=BUFFER)
breakdown = roi_grouped(r["bets"], ["league", "season"]).unstack("season").round(3)
print(f"ROI by league x season (b365), running map, {len(r['bets'])} bets:")
print(breakdown.to_string())

ROI by league x season (b365), running map, 997 bets:
season    2223   2324   2425   2526
league                             
england -0.120 -0.103  0.161  0.700
france   0.168 -0.001 -0.148  0.691
germany  0.126  0.557  0.193 -0.163
italy    0.184  0.289 -0.100 -0.194
spain    0.139  0.883 -0.361 -0.026


## Summary
The three gates at a glance for the deployed config (running map, k=5), and the one-line row written
to `model_checks/results.md`.

In [10]:
_ = run_check(matches, features=RUNNING_FEATURES, outcome=OUTCOME, implied=IMPLIED,
              segmenter=SEGMENTERS["running"], make_model=make_model, buffer=BUFFER,
              source="goals_foragst_draw_kmeans_match_lr", space="goals_foragst", target="draw",
              abt_filter="major, gn>8, dropna", segmentation=f"KMeans k={K} (match, 4D)",
              segment_filter="train-profitable clusters")

goals_foragst_draw_kmeans_match_lr   pooled +0.136 (997 bets)   [b365]

GATE 1  LOSO — ROI by test season           PASS  (+3/4 seasons)
    2223 +0.096    2324 +0.267    2425 -0.009    2526 +0.227

GATE 2  league — ROI by league / drop best   PASS
    france +0.189    germany +0.157    england +0.156    spain +0.078    italy +0.046
    pooled without best league (france): +0.124

GATE 3  walk-forward — ROI by test season    PASS
    2425 +0.059    2526 +0.227
